# Week 3 - CNN untuk MNIST
Marvel Kevin Nathanael | 00000108042

Implementasi LeNet sederhana dari TLM 3. Gambar tetap berbentuk 28x28 saat masuk CNN.

In [1]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

tf.keras.utils.set_random_seed(42)
(x_dev, y_dev), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_dev = x_dev.astype("float32")[..., None] / 255.0
x_test = x_test.astype("float32")[..., None] / 255.0
x_train, x_val, y_train, y_val = train_test_split(
    x_dev, y_dev, test_size=0.1, random_state=42, stratify=y_dev)
print(x_train.shape, x_val.shape, x_test.shape)

(54000, 28, 28, 1) (6000, 28, 28, 1) (10000, 28, 28, 1)


In [2]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(20, 5, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(50, 5, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()
history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                    epochs=8, batch_size=128, verbose=0,
                    callbacks=[tf.keras.callbacks.EarlyStopping(
                        monitor="val_loss", patience=2, restore_best_weights=True)])

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 20)     │           520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 20)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 50)       │        25,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 4, 4, 50)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       102,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

Total params: 129,388 (505.42 KB)

Trainable params: 129,388 (505.42 KB)

Non-trainable params: 0 (0.00 B)

In [3]:
loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
print(f"Test accuracy: {accuracy:.4f} | Test loss: {loss:.4f}")
print(confusion_matrix(y_test, y_pred))

Test accuracy: 0.9913 | Test loss: 0.0248
[[ 978    0    0    0    0    1    0    1    0    0]
 [   0 1129    1    0    0    1    1    2    1    0]
 [   2    0 1027    0    0    0    0    2    1    0]
 [   1    0    3 1000    0    4    0    0    2    0]
 [   1    0    1    0  973    0    1    0    1    5]
 [   2    0    0    5    0  880    2    0    2    1]
 [   8    2    0    1    1    3  941    0    2    0]
 [   0    1    2    1    0    0    0 1022    1    1]
 [   3    0    1    0    0    1    0    0  967    2]
 [   0    0    1    0    4    5    0    2    1  996]]


**Catatan:** Akurasi MNIST belum menjamin akurasi pada foto angka dari kamera ponsel.